# Lesson: Persistence e Human-in-the-Loop

Questa lezione copre i concetti di Persistence & Streaming
e Human in the Loop applicati al caso aziendale di acquisto hardware.

**Cosa costruiamo:**
1. La stessa classe `Agent` delle lezioni precedenti, estesa con un **checkpointer** (`MemorySaver`)
   per persistere lo stato tra le chiamate
2. Streaming con `thread_id` — lo stato sopravvive tra invocazioni dello stesso thread
3. Approvazione manuale con `interrupt_before` — il grafo si ferma prima di eseguire
   un nodo e aspetta l'input del manager
4. `get_state()`, `update_state()`, `get_state_history()` — ispezione e modifica dello stato


> **Nota pratica con modelli locali**: llama3.2 può perdere il filo su messaggi ambigui in thread lunghi. Includere sempre badge ID e codice prodotto nel messaggio garantisce che la procedura venga seguita correttamente. Con modelli più potenti (GPT-4, Claude) il multi-turn funziona anche con riferimenti impliciti come `"E se volessi anche il Dock?"`.

## 1. Setup

In [1]:
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated
from uuid import uuid4

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import (
    AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage
)
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

llm      = ChatOllama(model="llama3.2", temperature=0)
DATA_DIR = Path("data")

with open(DATA_DIR / "employees.json") as f: EMPLOYEES_DB = json.load(f)
with open(DATA_DIR / "products.json")  as f: PRODUCTS_DB  = json.load(f)
with open(DATA_DIR / "policy_hardware.md") as f: POLICY_TEXT = f.read()

print(f"✅ Setup — dipendenti: {len(EMPLOYEES_DB)}, prodotti: {len(PRODUCTS_DB)}")


✅ Setup — dipendenti: 3, prodotti: 4


## 2. Tool del dominio

In [2]:
# Registro in-session degli importi già impegnati (resettato a ogni riavvio del kernel)
# Chiave: employee_id, valore: totale EUR già approvati nella sessione corrente
_budget_impegnato: dict[str, float] = {}


@tool
def get_budget_dipendente(employee_id: str) -> str:
    """Restituisce budget residuo (al netto degli acquisti già approvati in sessione),
    ruolo e manager del dipendente.

    Args:
        employee_id: Badge ID del dipendente (es. 'E4521')
    """
    emp = EMPLOYEES_DB.get(employee_id)
    if not emp:
        return f"Dipendente '{employee_id}' non trovato nel sistema."
    budget_originale = emp.get("budget_residuo", 0)
    gia_impegnato    = _budget_impegnato.get(employee_id, 0)
    budget_residuo   = budget_originale - gia_impegnato
    return (
        f"Dipendente: {emp['name']} ({emp['role']}, {emp['department']})\n"
        f"Budget originale: {budget_originale} EUR\n"
        f"Già impegnato in sessione: {gia_impegnato} EUR\n"
        f"Budget residuo attuale: {budget_residuo} EUR\n"
        f"Progetto attivo: {emp.get('active_project', 'N/A')}\n"
        f"Manager: {emp['manager_email']}"
    )


@tool
def cerca_prodotti_catalogo(query: str) -> str:
    """Cerca prodotti nel catalogo aziendale per nome o categoria.

    Funziona anche con query multi-parola: basta che UNA parola della query
    sia contenuta nel nome o nella categoria del prodotto.

    Args:
        query: Termine di ricerca, anche multi-parola (es. 'thunderbolt dock', 'monitor 4k')
    """
    stopwords = {"per", "del", "con", "un", "il", "la", "di", "da", "in", "e"}
    keywords  = [w for w in query.lower().split() if len(w) >= 2 and w not in stopwords]
    risultati = {
        pid: p for pid, p in PRODUCTS_DB.items()
        if any(
            kw in p.get("name", "").lower() or kw in p.get("category", "").lower()
            for kw in keywords
        )
    }
    if not risultati:
        return f"Nessun prodotto trovato per '{query}'."
    righe = [f"Trovati {len(risultati)} prodotto/i per '{query}':"]
    for pid, p in risultati.items():
        righe.append(
            f"  [{pid}] {p['name']} — {p['price_eur']} EUR ({p['category']})"
        )
    return "\n".join(righe)


@tool
def valuta_acquisto(employee_id: str, prodotto_id: str) -> str:
    """Valuta se un acquisto è approvabile tenendo conto del budget già impegnato
    in questa sessione (acquisti precedentemente approvati nel thread corrente).

    Se approvato, registra l'importo come impegnato per i controlli successivi.

    Applica le regole della policy:
    - APPROVATO: prezzo <= budget residuo E (laptop OPPURE monitor/accessorio <= 300 EUR)
    - ESCALATION: monitor/accessorio > 300 EUR OPPURE prezzo > budget residuo sessione
    - RIFIUTO: dipendente o prodotto non trovati

    Args:
        employee_id: Badge ID del dipendente (es. 'E4521')
        prodotto_id: Codice prodotto dal catalogo (es. 'LAP-DEV-01', 'ACC-DOCK-01')
    """
    emp = EMPLOYEES_DB.get(employee_id)
    if not emp:
        return f"RIFIUTO — Dipendente '{employee_id}' non trovato."

    prod = PRODUCTS_DB.get(prodotto_id)
    if not prod:
        match = next(
            (p for pid, p in PRODUCTS_DB.items()
             if prodotto_id.lower() in pid.lower()
             or prodotto_id.lower() in p.get("name", "").lower()),
            None
        )
        if not match:
            return f"RIFIUTO — Prodotto '{prodotto_id}' non trovato nel catalogo."
        prod = match

    budget_originale  = emp.get("budget_residuo", 0)
    gia_impegnato     = _budget_impegnato.get(employee_id, 0)
    budget_residuo    = budget_originale - gia_impegnato
    prezzo            = prod.get("price_eur", 0)
    categoria         = prod.get("category", "").lower()
    nome              = prod.get("name", prodotto_id)

    if categoria in ("monitor", "accessory") and prezzo > 300:
        return (
            f"ESCALATION — {nome} ({prezzo} EUR) supera la soglia di 300 EUR "
            f"per monitor/accessori. Contattare: {emp['manager_email']}"
        )
    if prezzo > budget_residuo:
        return (
            f"ESCALATION — {nome} ({prezzo} EUR) supera il budget residuo "
            f"di sessione {budget_residuo} EUR "
            f"(originale {budget_originale} EUR, già impegnato {gia_impegnato} EUR). "
            f"Contattare: {emp['manager_email']}"
        )

    # Acquisto approvato: registra l'importo come impegnato
    _budget_impegnato[employee_id] = gia_impegnato + prezzo
    nuovo_residuo = budget_originale - _budget_impegnato[employee_id]
    return (
        f"APPROVATO — {emp['name']} ({emp['role']}) può acquistare {nome}: "
        f"{prezzo} EUR. "
        f"Budget residuo aggiornato: {nuovo_residuo} EUR "
        f"(impegnato in sessione: {_budget_impegnato[employee_id]} EUR)."
    )


tools_acquisto = [get_budget_dipendente, cerca_prodotti_catalogo, valuta_acquisto]
print("✅ Tool:", [t.name for t in tools_acquisto])
print("   _budget_impegnato (sessione):", _budget_impegnato)


✅ Tool: ['get_budget_dipendente', 'cerca_prodotti_catalogo', 'valuta_acquisto']
   _budget_impegnato (sessione): {}


## 3. La classe `Agent` con Checkpointer

Aggiungiamo il **checkpointer** (`MemorySaver`):
- Lo stato viene **salvato** dopo ogni step
- Chiamate successive con lo stesso `thread_id` **riprendono** dal punto precedente
- Il `thread_id` funge da "sessione" — thread diversi sono conversazioni indipendenti


In [3]:
class Agent:
    """Agente ReAct con supporto per checkpointer."""

    def __init__(self, model, tools, checkpointer=None, system=""):
        self.system = system

        class AgentState(TypedDict):
            messages: Annotated[list[AnyMessage], operator.add]

        graph = StateGraph(AgentState)
        graph.add_node("llm",    self.call_llm)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        # compile con checkpointer opzionale
        self.graph = graph.compile(checkpointer=checkpointer)

        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state):
        return len(state["messages"][-1].tool_calls) > 0

    def call_llm(self, state):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"  🔧 {t['name']}({t['args']})")
            if t["name"] not in self.tools:
                result = f"Tool '{t['name']}' non esiste. Disponibili: {list(self.tools.keys())}"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(
                tool_call_id=t["id"], name=t["name"], content=str(result)
            ))
        return {"messages": results}


print("✅ Classe Agent definita")


✅ Classe Agent definita


## 4. Persistence e Multi-turn con `thread_id`

Passiamo `{"configurable": {"thread_id": "..."}}` a ogni chiamata.
Lo stesso thread mantiene la memoria della conversazione.
Thread diversi sono sessioni indipendenti.

In [4]:
memory = MemorySaver()

system_prompt = (
    "Sei l'Assistente Acquisti Hardware aziendale. "
    "Ricevi richieste di acquisto da dipendenti identificati dal loro badge ID.\n"
    "Segui SEMPRE questo ordine:\n"
    "1. Chiama get_budget_dipendente con il badge ID (es. 'E4521') "
    "per ottenere budget residuo e ruolo.\n"
    "2. Chiama cerca_prodotti_catalogo con UNA sola parola chiave del prodotto "
    "(es. 'thinkpad', 'monitor', 'laptop') per trovare il codice prodotto.\n"
    "3. Chiama valuta_acquisto con il badge ID e il CODICE PRODOTTO trovato al passo 2 "
    "(es. 'LAP-DEV-01', 'MON-4K-01') — NON passare il prezzo, viene letto automaticamente.\n"
    "4. Rispondi in italiano con la decisione (APPROVATO / ESCALATION / RIFIUTO) e motivazione."
)

abot = Agent(
    model=llm,
    tools=tools_acquisto,
    checkpointer=memory,
    system=system_prompt
)
print("✅ Agente con MemorySaver pronto")


✅ Agente con MemorySaver pronto


In [5]:
# Prima domanda — thread "acquisto-001"
messages = [HumanMessage(content="Ciao, sono Mario Rossi (E4521). Posso acquistare un ThinkPad P16s?")]
thread   = {"configurable": {"thread_id": "acquisto-001"}}

for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        last = v["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(last.content)


  🔧 get_budget_dipendente({'employee_id': 'E4521'})
  🔧 cerca_prodotti_catalogo({'query': 'thinkpad'})
  🔧 valuta_acquisto({'employee_id': 'E4521', 'prodotto_id': 'LAP-DEV-01'})
Ciao Mario Rossi,

La tua richiesta di acquisto di ThinkPad P16s Gen 2 / MacBook Pro M3 è stata approvata. Il budget residuo attuale è di 150 EUR, poiché è stato impegnato in sessione 1650 EUR.

Se hai bisogno di ulteriore assistenza o hai altre richieste di acquisto, non esitare a chiedere.

Cordiali saluti,
[Il mio nome]


In [6]:
# Seconda domanda sullo STESSO thread — l'agente ricorda la sessione
# Il ThinkPad (1650 EUR) è già stato approvato: budget residuo sessione = 1800 - 1650 = 150 EUR
# Il Dock costa 190 EUR > 150 EUR residui → atteso ESCALATION
messages = [HumanMessage(content=(
    "Sempre Mario Rossi (E4521) — vorrei anche il Thunderbolt 4 Dock (ACC-DOCK-01)."
))]
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        last = v["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(last.content)


  🔧 get_budget_dipendente({'employee_id': 'E4521'})
  🔧 cerca_prodotti_catalogo({'query': 'thunderbolt dock'})
  🔧 valuta_acquisto({'employee_id': 'E4521', 'prodotto_id': 'ACC-DOCK-01'})
Ciao Mario Rossi,

La tua richiesta di acquisto del Thunderbolt 4 Universal Dock è stata escalata a causa del budget residuo attuale. Il dock ha un prezzo di 190 EUR, che supera il budget residuo di 150 EUR.

Per procedere con l'acquisto, ti consiglio di contattare il nostro manager, Chiara Bianchi, al numero di telefono [inserisci numero di telefono] o via email a chiara.bianchi@azienda.it.

Spero che questo ti sia stato utile. Se hai altre richieste di acquisto, non esitare a chiedere.

Cordiali saluti,
[Il mio nome]


In [7]:
# Thread DIVERSO — sessione indipendente, nessuna memoria del thread precedente
# Includiamo badge ID e codice prodotto per guidare llama3.2 alla valutazione diretta
messages2 = [HumanMessage(content=(
    "Sono Giulia Verdi (E1082). "
    "Voglio acquistare il monitor Dell UltraSharp 27 4K (codice MON-4K-01)."
))]
thread2 = {"configurable": {"thread_id": "acquisto-002"}}

for event in abot.graph.stream({"messages": messages2}, thread2):
    for v in event.values():
        last = v["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(last.content)


  🔧 valuta_acquisto({'employee_id': 'E1082', 'prodotto_id': 'MON-4K-01'})
Ciao Giulia Verdi,

La tua richiesta di acquisto del monitor Dell UltraSharp 27 4K (codice MON-4K-01) è stata valutata e, purtroppo, è stata classificata come ESCALATION.

Il prezzo di questo monitor, pari a 520 EUR, supera la soglia di 300 EUR per monitor/accessori stabilita dalla nostra azienda. Pertanto, non possiamo approvare la tua richiesta di acquisto.

Se desideri discutere ulteriormente della tua richiesta o se hai altre domande, non esitare a contattarmi.

Cordiali saluti,
[Il mio Nome]


## 5. Ispezione dello Stato — `get_state()` e `get_state_history()`

Possiamo leggere lo stato salvato dal checkpointer
in qualsiasi momento, ispezionare la cronologia e capire il flusso eseguito.

In [8]:
# Stato corrente del thread "acquisto-001"
snapshot = abot.graph.get_state(thread)

print(f"Nodo successivo (next): {snapshot.next}")
print(f"Numero messaggi:        {len(snapshot.values['messages'])}")
print()
print("Cronologia messaggi:")
for msg in snapshot.values["messages"]:
    tipo = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        calls = [tc["name"] for tc in msg.tool_calls]
        print(f"  [{tipo}] tool_calls={calls}")
    elif hasattr(msg, "name") and msg.name:
        preview = str(msg.content)[:80].replace("\n", " ")
        print(f"  [{tipo}:{msg.name}] {preview}")
    else:
        preview = str(msg.content)[:80].replace("\n", " ")
        print(f"  [{tipo}] {preview}")


Nodo successivo (next): ()
Numero messaggi:        12

Cronologia messaggi:
  [HumanMessage] Ciao, sono Mario Rossi (E4521). Posso acquistare un ThinkPad P16s?
  [AIMessage] tool_calls=['get_budget_dipendente', 'cerca_prodotti_catalogo', 'valuta_acquisto']
  [ToolMessage:get_budget_dipendente] Dipendente: Mario Rossi (Senior Developer, Engineering) Budget originale: 1800 E
  [ToolMessage:cerca_prodotti_catalogo] Trovati 1 prodotto/i per 'thinkpad':   [LAP-DEV-01] ThinkPad P16s Gen 2 / MacBoo
  [ToolMessage:valuta_acquisto] APPROVATO — Mario Rossi (Senior Developer) può acquistare ThinkPad P16s Gen 2 / 
  [AIMessage] Ciao Mario Rossi,  La tua richiesta di acquisto di ThinkPad P16s Gen 2 / MacBook
  [HumanMessage] Sempre Mario Rossi (E4521) — vorrei anche il Thunderbolt 4 Dock (ACC-DOCK-01).
  [AIMessage] tool_calls=['get_budget_dipendente', 'cerca_prodotti_catalogo', 'valuta_acquisto']
  [ToolMessage:get_budget_dipendente] Dipendente: Mario Rossi (Senior Developer, Engineering) Budget o

In [9]:
# Cronologia di tutti gli stati del thread "acquisto-001"
print("State history (più recente prima):")
for i, state in enumerate(abot.graph.get_state_history(thread)):
    n_msg = len(state.values.get("messages", []))
    print(f"  step={state.metadata.get('step','?'):>2} | "
          f"next={str(state.next):<25} | messaggi={n_msg}")


State history (più recente prima):
  step= 8 | next=()                        | messaggi=12
  step= 7 | next=('llm',)                  | messaggi=11
  step= 6 | next=('action',)               | messaggi=8
  step= 5 | next=('llm',)                  | messaggi=7
  step= 4 | next=('__start__',)            | messaggi=6
  step= 3 | next=()                        | messaggi=6
  step= 2 | next=('llm',)                  | messaggi=5
  step= 1 | next=('action',)               | messaggi=2
  step= 0 | next=('llm',)                  | messaggi=1
  step=-1 | next=('__start__',)            | messaggi=0


## 6. Human-in-the-Loop con `interrupt_before`

Compiliamo il grafo con `interrupt_before=["action"]`.
Il grafo si **ferma prima di eseguire il nodo `action`** (prima di chiamare i tool)
e aspetta che l'operatore umano decida se procedere.

```
llm → [INTERRUPT] → action → llm → END
         ↑
    il manager vede cosa sta per fare il LLM
    e decide se autorizzare
```

Per **riprendere**: chiamiamo `.stream(None, thread)` — `None` come input
segnala al grafo di continuare dall'interruzione.

In [10]:
# Nuovo agente con interrupt_before — stessa classe Agent, diversa compilazione
class AgentConApprovazione(Agent):
    """Estende Agent compilando con interrupt_before=['action']."""

    def __init__(self, model, tools, checkpointer=None, system=""):
        # Chiamiamo __init__ del parent, poi ricompiliamo con interrupt_before
        super().__init__(model, tools, checkpointer=None, system=system)

        # Ricostruiamo AgentState localmente per la ricompilazione
        class AgentState(TypedDict):
            messages: Annotated[list[AnyMessage], operator.add]

        graph = StateGraph(AgentState)
        graph.add_node("llm",    self.call_llm)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm", self.exists_action, {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")

        # ⚡ interrupt_before — il grafo si ferma PRIMA di eseguire 'action'
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)


memory_approval = MemorySaver()
abot_approval   = AgentConApprovazione(
    model=llm,
    tools=tools_acquisto,
    checkpointer=memory_approval,
    system=system_prompt
)
print("✅ Agente con interrupt_before=['action'] pronto")


✅ Agente con interrupt_before=['action'] pronto


In [11]:
# Prima esecuzione — il grafo si ferma prima di chiamare i tool
messages = [HumanMessage(content="Giulia Verdi (E1082) vuole il monitor Dell UltraSharp (MON-4K-01).")]
thread_a = {"configurable": {"thread_id": "approval-001"}}

print("=== Prima esecuzione (andrà in interrupt) ===")
for event in abot_approval.graph.stream({"messages": messages}, thread_a):
    for node, v in event.items():
        # LangGraph emette un evento speciale {'__interrupt__': (<Interrupt>,)} quando
        # interrupt_before scatta — v è una tupla, non un dict con "messages"
        if node == "__interrupt__":
            print("\n⏸️  INTERRUPT — il LLM vuole chiamare i tool")
            print("   → stream(None, thread_a) per autorizzare")
            continue
        # Evento normale: v è un dict con "messages"
        if not isinstance(v, dict) or "messages" not in v:
            continue
        last = v["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            print("\n⏸️  INTERRUPT — il LLM vuole chiamare:")
            for tc in last.tool_calls:
                print(f"   🔧 {tc['name']}({tc['args']})")
            print("   → Il manager decide se autorizzare con stream(None, thread_a)")
        elif isinstance(last, AIMessage) and last.content:
            print(last.content)


=== Prima esecuzione (andrà in interrupt) ===



⏸️  INTERRUPT — il LLM vuole chiamare:
   🔧 valuta_acquisto({'prodotto_id': 'MON-4K-01', 'employee_id': 'E1082'})
   → Il manager decide se autorizzare con stream(None, thread_a)

⏸️  INTERRUPT — il LLM vuole chiamare i tool
   → stream(None, thread_a) per autorizzare


In [12]:
# Stato dopo l'interrupt — next indica il nodo in attesa
stato_interrupt = abot_approval.graph.get_state(thread_a)
print(f"Nodo in attesa (next): {stato_interrupt.next}")
print(f"Tool calls pianificate: "
      f"{[tc['name'] for tc in stato_interrupt.values['messages'][-1].tool_calls]}")


Nodo in attesa (next): ('action',)
Tool calls pianificate: ['valuta_acquisto']


In [13]:
# Il manager approva — riprendiamo con stream(None, thread)
print("=== Il manager approva — ripresa ===")
for event in abot_approval.graph.stream(None, thread_a):
    for node, v in event.items():
        if node == "__interrupt__":
            continue
        if not isinstance(v, dict) or "messages" not in v:
            continue
        last = v["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(last.content)
        elif hasattr(last, "name") and last.name:
            print(f"  ↩️  [{last.name}] {str(last.content)[:100]}")


=== Il manager approva — ripresa ===
  🔧 valuta_acquisto({'prodotto_id': 'MON-4K-01', 'employee_id': 'E1082'})
  ↩️  [valuta_acquisto] ESCALATION — Dell UltraSharp 27 4K (U2723QE) (520 EUR) supera la soglia di 300 EUR per monitor/acces
Ciao Giulia Verdi,

Mi dispiace ma il monitor Dell UltraSharp (MON-4K-01) che hai richiesto è stato classificato come "ESCALATION" a causa del suo prezzo superiore alla soglia di 300 EUR. Ci dispiace non poter approvarla.

Per favore, contatta Luca Neri al luca.neri@azienda.it per ulteriori informazioni e possibili alternative.

Scusa per la delusione.


## 7. Modificare lo Stato — `update_state()`

Possiamo **modificare le tool call** che il LLM ha pianificato
prima che vengano eseguite.
Caso d'uso: il manager vuole autorizzare ma con un prodotto diverso.

In [14]:
# Nuova richiesta — thread diverso
messages = [HumanMessage(content="Mario Rossi (E4521) chiede un laptop per sviluppo.")]
thread_b  = {"configurable": {"thread_id": "approval-002"}}

for event in abot_approval.graph.stream({"messages": messages}, thread_b):
    for node, v in event.items():
        if node == "__interrupt__":
            print("\n⏸️  INTERRUPT — il LLM vuole chiamare i tool")
            print("   → stream(None, thread_b) per autorizzare")
            continue
        if not isinstance(v, dict) or "messages" not in v:
            continue
        last = v["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            print("⏸️  INTERRUPT — tool calls pianificate:")
            for tc in last.tool_calls:
                print(f"   {tc['name']}({tc['args']})")


⏸️  INTERRUPT — tool calls pianificate:
   get_budget_dipendente({'employee_id': 'E4521'})

⏸️  INTERRUPT — il LLM vuole chiamare i tool
   → stream(None, thread_b) per autorizzare


In [15]:
# Leggiamo lo stato corrente
current = abot_approval.graph.get_state(thread_b)
print("Tool call corrente:")
print(current.values["messages"][-1].tool_calls)


Tool call corrente:
[{'name': 'get_budget_dipendente', 'args': {'employee_id': 'E4521'}, 'id': 'a0753548-53c3-4b49-b4bf-091182ac6f81', 'type': 'tool_call'}]


In [16]:
# Modifichiamo la query di ricerca prima che venga eseguita
# (es. il manager vuole cercare 'thinkpad' invece del termine generico)
_id = current.values["messages"][-1].tool_calls[0]["id"]

current.values["messages"][-1].tool_calls = [{
    "name": "cerca_prodotti_catalogo",
    "args": {"query": "thinkpad"},
    "id":   _id,
    "type": "tool_call"
}]

abot_approval.graph.update_state(thread_b, current.values)
print("✅ Stato aggiornato — tool call modificata in cerca_prodotti_catalogo('thinkpad')")


✅ Stato aggiornato — tool call modificata in cerca_prodotti_catalogo('thinkpad')


In [17]:
# Riprendiamo — ora esegue la tool call modificata
for event in abot_approval.graph.stream(None, thread_b):
    for node, v in event.items():
        if node == "__interrupt__":
            continue
        if not isinstance(v, dict) or "messages" not in v:
            continue
        last = v["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(last.content)
        elif hasattr(last, "name") and last.name:
            print(f"  [{last.name}] {str(last.content)[:120]}")


  🔧 cerca_prodotti_catalogo({'query': 'thinkpad'})
  [cerca_prodotti_catalogo] Trovati 1 prodotto/i per 'thinkpad':
  [LAP-DEV-01] ThinkPad P16s Gen 2 / MacBook Pro M3 — 1650 EUR (laptop)


## 8. Time Travel — `get_state_history()` e Replay

Possiamo tornare a un punto precedente della conversazione
e rieseguire da lì — creando un **branch** alternativo.

In [18]:
# Visualizziamo la storia completa del thread approval-001
print("State history di 'approval-001' (più recente prima):")
states = []
for state in abot_approval.graph.get_state_history(thread_a):
    states.append(state)
    n = len(state.values.get("messages", []))
    print(f"  step={state.metadata.get('step','?'):>2} | "
          f"next={str(state.next):<20} | messaggi={n}")


State history di 'approval-001' (più recente prima):
  step= 3 | next=()                   | messaggi=4
  step= 2 | next=('llm',)             | messaggi=3
  step= 1 | next=('action',)          | messaggi=2
  step= 0 | next=('llm',)             | messaggi=1
  step=-1 | next=('__start__',)       | messaggi=0


In [19]:
# Torniamo allo stato prima dell'interrupt (step iniziale)
# e rieseguiamo — questa volta il manager modifica la richiesta
to_replay = states[-2]   # stato iniziale con solo il messaggio utente
print(f"Stato da riprendere: step={to_replay.metadata.get('step')}, next={to_replay.next}")
print(f"Messaggi: {len(to_replay.values['messages'])}")


Stato da riprendere: step=0, next=('llm',)
Messaggi: 1


In [ ]:
# Rieseguiamo dal punto precedente — branch alternativo
# Cambiamo la query utente per simulare una richiesta diversa
new_messages = to_replay.values["messages"].copy()
new_messages[-1] = HumanMessage(content="Giulia Verdi (E1082) chiede un laptop standard, non il monitor.")

branch_config = abot_approval.graph.update_state(
    to_replay.config,
    {"messages": new_messages},
    as_node="__start__"
)
print("Branch creato — riesecuzione:")
for event in abot_approval.graph.stream(None, branch_config):
    for node, v in event.items():
        if node == "__interrupt__":
            print("\n⏸️  interrupt nel branch — tool calls pianificate")
            continue
        if not isinstance(v, dict) or "messages" not in v:
            continue
        last = v["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(last.content[:200])
        elif hasattr(last, "tool_calls") and last.tool_calls:
            print(f"  ⏸️  interrupt: {[tc['name'] for tc in last.tool_calls]}")


Branch creato — riesecuzione:
```json
{
  "name": "valuta_acquisto",
  "parameters": {
    "employee_id": "E1082",
    "prodotto_id": "MON-4K-01"
  }
}
```

Poi, per rispondere alla richiesta di Giulia Verdi, dovrei chiamare `cerc


: 

## 9. Riepilogo

| Concetto | Parte 1 | Parte 2 |
|---|---|---|
| **Checkpointer** | `MemorySaver` passato a `compile()` | Stesso |
| **Multi-turn** | Stesso `thread_id` → memoria persistente | Stesso |
| **Interrupt** | — | `interrupt_before=["action"]` in `compile()` |
| **Riprendere** | — | `stream(None, thread)` |
| **Modificare stato** | — | `update_state(thread, values)` |
| **Time travel** | — | `get_state_history()` + `update_state(old_config, ...)` |

**Pattern chiave:**
```python
# 1. Prima esecuzione — si ferma all'interrupt
for event in abot.graph.stream({"messages": [msg]}, thread): ...

# 2. Ispeziona cosa sta per fare il LLM
stato = abot.graph.get_state(thread)   # next = ("action",)

# 3. Eventualmente modifica le tool calls
abot.graph.update_state(thread, nuovo_stato)

# 4. Riprendi
for event in abot.graph.stream(None, thread): ...
```

## 10. Esercizi

### Esercizio A — Flusso interattivo con input reale
Costruisci una funzione `gestisci_richiesta(richiesta, employee_id, thread_id)` che:
1. Avvia il grafo con `stream()`
2. Se intercetta un interrupt, stampa le tool call pianificate e chiede all'operatore
   `input("Autorizzare? [s/n]: ")`
3. Se l'operatore risponde `s`, riprende con `stream(None, thread)`
4. Se risponde `n`, chiama `update_state` per svuotare le tool calls e termina

### Esercizio B — Rifiuto automatico fuori orario
Aggiungi un tool `controlla_orario()` che restituisce se siamo in orario lavorativo
(9:00–18:00). Modifica il system prompt affinché il LLM chiami questo tool prima di
procedere, e imposta `interrupt_before=["action"]` solo fuori orario.